# 5.2 — Clasificación jerárquica con features clínicas enriquecidas

**Objetivo**: Evaluar si la incorporación de variables clínicas y etiquetas de cluster
como **features en X** mejora la clasificación jerárquica de dos etapas.

**Etapa 1**: Detección binaria (cancer vs nonMalignant) — con features enriquecidas.
**Etapa 2**: Clasificación del tipo de cáncer (18 tipos) — con features enriquecidas.

**Variable objetivo (y)**: 19 clases originales (18 tipos de cáncer + nonMalignant).

**Features en X**: ~5440 genes + mal_cluster dummies + nm_cluster dummies + Age + Sex.

**Prevención de data leakage**: `ClinicalFeaturePreprocessor` dentro del pipeline
se re-ajusta por fold en CV y en el fit final.

**Comparación**: vs 5.0 (solo genes) y 5.1 (clusters en y).

In [2]:
from __future__ import annotations
from pathlib import Path
import logging
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from time import perf_counter
from tqdm.auto import tqdm

from genomics_dl.models.train_multiclass import (
    HierarchicalTrainConfig,
    run_hierarchical_training,
)

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("mlflow").setLevel(logging.WARNING)

### Rutas y carga de datos

In [3]:
DATA_PROCESSED = Path("../data/processed")

TRAIN_PATH = DATA_PROCESSED / "gse183635_tep_tpm_train_clustered.parquet"
TEST_PATH  = DATA_PROCESSED / "gse183635_tep_tpm_test_clustered.parquet"

df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f"Train: {df_train.shape}")
print(f"Test:  {df_test.shape}")

Train: (1880, 5454)
Test:  (471, 5454)


### Separación genes vs metadatos

In [4]:
metadata_cols = [
    "Sample ID", "Patient_group", "Stage", "Sex", "Age",
    "Sample-supplying institution", "Training series",
    "Evaluation series", "Validation series", "lib.size",
    "classificationScoreCancer", "Class_group",
    "mal_cluster", "nm_cluster",
]

gene_cols = [c for c in df_train.columns if c not in metadata_cols]
print(f"Genes: {len(gene_cols)}")
print(f"Columnas clínicas/cluster disponibles: mal_cluster, nm_cluster, Age, Sex")

Genes: 5440
Columnas clínicas/cluster disponibles: mal_cluster, nm_cluster, Age, Sex


### Sanity check de etiquetas (train)

In [5]:
df_train["Class_group"].value_counts(dropna=False)

Class_group
Malignant       1302
nonMalignant     578
Name: count, dtype: int64

In [6]:
df_train.loc[
    df_train["Class_group"].astype(str) == "Malignant", "Patient_group"
].value_counts().head(20)

Patient_group
Non-small-cell lung cancer    417
Ovarian cancer                114
Glioma                        113
Pancreatic cancer              93
Breast cancer                  80
Head and neck cancer           79
Cholangiocarcinoma             71
Colorectal cancer              69
Melanoma                       54
Sarcoma                        44
Endometrial cancer             34
Prostate cancer                23
Multiple Myeloma               22
Urothelial cancer              22
Renal cell cancer              20
Hepatocellular carcinoma       19
Lymphoma                       16
Esophageal carcinoma           12
Name: count, dtype: int64

## Experimentos jerárquicos con features clínicas (sweep)

Ranking:
1) Minimizar `test_cancer_fn`
2) Maximizar `test_cancer_recall_sensitivity`
3) Maximizar `test_f1_macro`

In [7]:
def slugify_token(value):
    return str(value).replace(".", "p").replace("-", "m")

def fmt_secs(s: float) -> str:
    s = int(max(0, s))
    h = s // 3600
    m = (s % 3600) // 60
    ss = s % 60
    if h > 0:
        return f"{h:d}h {m:02d}m {ss:02d}s"
    if m > 0:
        return f"{m:d}m {ss:02d}s"
    return f"{ss:d}s"

In [8]:
# Configuración base con features clínicas
BASE_CONFIG = dict(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    nonmalignant_label="nonMalignant",
    malignant_label="Malignant",
    use_pca=False,
    var_quantile=0.15,
    selector_on_log=False,
    variance_filter_threshold=1e-6,
    cv_splits=5,
    random_state=42,
    experiment_name="gse183635_hierarchical_clinical",
    save_local_bundle=False,
    save_plots=False,
    mlflow_log_artifacts=False,
    mlflow_log_model=False,
    # --- Features clínicas ---
    cluster_as_feature=True,
    malignant_cluster_col="mal_cluster",
    nm_cluster_col="nm_cluster",
    age_col="Age",
    sex_col="Sex",
)

# Clasificadores etapa 1 (detección binaria)
STAGE1_CLASSIFIERS = [
    ("xgboost", dict(n_estimators=400, max_depth=6, learning_rate=0.1)),
    ("lightgbm", dict(n_estimators=400, max_depth=6, learning_rate=0.1)),
]

# Clasificadores etapa 2 (tipo de cáncer)
STAGE2_CLASSIFIERS = [
    ("xgboost", dict(n_estimators=400, max_depth=8, learning_rate=0.05)),
    ("lightgbm", dict(n_estimators=400, max_depth=8, learning_rate=0.05)),
]

# Diccionarios para modelo final (más estimadores)
STAGE1_CLASSIFIERS_DICT = {
    "xgboost": dict(n_estimators=650, max_depth=6, learning_rate=0.1),
    "lightgbm": dict(n_estimators=650, max_depth=6, learning_rate=0.1),
}
STAGE2_CLASSIFIERS_DICT = {
    "xgboost": dict(n_estimators=800, max_depth=8, learning_rate=0.05),
    "lightgbm": dict(n_estimators=800, max_depth=8, learning_rate=0.05),
}

# Estrategias de pesos por clase para etapa 2
STAGE2_WEIGHTINGS = ["balanced", "sqrt", "log"]

# Recall mínimo para etapa 1
STAGE1_MIN_RECALLS = [0.90]

# Construir sweep
sweep = []
for s1_name, s1_params in STAGE1_CLASSIFIERS:
    for s2_name, s2_params in STAGE2_CLASSIFIERS:
        for s2_weighting in STAGE2_WEIGHTINGS:
            for s1_min_recall in STAGE1_MIN_RECALLS:
                sweep.append(dict(
                    s1_name=s1_name,
                    s1_params=s1_params,
                    s2_name=s2_name,
                    s2_params=s2_params,
                    s2_weighting=s2_weighting,
                    s1_min_recall=s1_min_recall,
                ))

print(f"Total combinaciones: {len(sweep)}")

Total combinaciones: 12


In [9]:
results = []
errors = []

total = len(sweep)
start_all = perf_counter()

ema = None
alpha = 0.25
done = 0

pbar = tqdm(sweep, total=total, desc="Sweep Jerárquico Clinical", unit="run")

for combo in pbar:
    t0 = perf_counter()

    s1_name = combo["s1_name"]
    s1_params = combo["s1_params"]
    s2_name = combo["s2_name"]
    s2_params = combo["s2_params"]
    s2_weighting = combo["s2_weighting"]
    s1_min_recall = combo["s1_min_recall"]

    model_name = f"hier_clinical_{s1_name}_{s2_name}_{s2_weighting}_mr{slugify_token(s1_min_recall)}"
    model_version = "v0.5.0"

    cfg = HierarchicalTrainConfig(
        **{
            **BASE_CONFIG,
            "model_name": model_name,
            "model_version": model_version,
            "stage1_clf_name": s1_name,
            "stage1_clf_params": s1_params,
            "stage1_min_recall": s1_min_recall,
            "stage2_clf_name": s2_name,
            "stage2_clf_params": s2_params,
            "stage2_class_weighting": s2_weighting,
            "skip_cv_for_sweep": True,
        }
    )

    try:
        out = run_hierarchical_training(cfg, feature_cols=gene_cols)
        tm = out["test_metrics"]
        results.append({
            "model_name": model_name,
            "s1_clf": s1_name,
            "s2_clf": s2_name,
            "s2_weighting": s2_weighting,
            "s1_min_recall": s1_min_recall,
            "test_cancer_fn": tm["cancer_fn"],
            "test_cancer_fnr": tm["cancer_fnr"],
            "test_cancer_recall": tm["cancer_recall_sensitivity"],
            "test_cancer_specificity": tm["cancer_specificity"],
            "test_f1_macro": tm["f1_macro"],
            "test_balanced_accuracy": tm["balanced_accuracy"],
            "test_accuracy": tm["accuracy"],
            "test_cancer_roc_auc": tm.get("cancer_roc_auc", np.nan),
            "mlflow_run_id": out["mlflow_run_id"],
        })
    except Exception as e:
        errors.append({
            "model_name": model_name,
            "s1_clf": s1_name,
            "s2_clf": s2_name,
            "error": repr(e),
        })

    dt = perf_counter() - t0
    ema = dt if ema is None else (alpha * dt + (1 - alpha) * ema)

    done += 1
    elapsed = perf_counter() - start_all
    remaining = (total - done) * (ema if ema is not None else 0.0)

    pbar.set_postfix({
        "last": fmt_secs(dt),
        "avg": fmt_secs(ema),
        "elapsed": fmt_secs(elapsed),
        "eta": fmt_secs(remaining),
        "ok": len(results),
        "err": len(errors),
    })

# DataFrames finales
res_df = (
    pd.DataFrame(results)
      .sort_values(
          ["test_cancer_fn", "test_cancer_fnr", "test_cancer_recall", "test_f1_macro"],
          ascending=[True, True, False, False],
      )
      .reset_index(drop=True)
)

err_df = pd.DataFrame(errors).reset_index(drop=True)

print(f"OK: {len(res_df)}, Errores: {len(err_df)}")

Sweep Jerárquico Clinical:   0%|          | 0/12 [00:00<?, ?run/s]2026/04/11 12:46:33 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/04/11 12:46:33 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/04/11 12:46:33 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/04/11 12:46:33 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/04/11 12:46:33 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/04/11 12:46:33 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/04/11 12:46:34 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/04/11 12:46:34 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/04/11 12:46:36 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/04/11 12:46:36 INFO alembic.runtime.migration: Will assume non-transactional DDL.
Sweep Jerárquico Clinical:  25%|██▌       | 3/12 [1

OK: 12, Errores: 0


### Resultados del sweep

In [10]:
display(res_df)

,model_name,s1_clf,s2_clf,s2_weighting,s1_min_recall,test_cancer_fn,test_cancer_fnr,test_cancer_recall,test_cancer_specificity,test_f1_macro,test_balanced_accuracy,test_accuracy,test_cancer_roc_auc,mlflow_run_id
0,hier_clinical_xgboost_lightgbm_balanced_mr0p9,xgboost,lightgbm,balanced,0.9,0,0.0,1.0,1.0,0.414269,0.391031,0.687898,1.0,55f050f21f124ff396029170266a0304
1,hier_clinical_lightgbm_lightgbm_balanced_mr0p9,lightgbm,lightgbm,balanced,0.9,0,0.0,1.0,1.0,0.414269,0.391031,0.687898,1.0,0f76b4eb64834f9aa5519ec890153684
2,hier_clinical_xgboost_lightgbm_log_mr0p9,xgboost,lightgbm,log,0.9,0,0.0,1.0,1.0,0.382967,0.365443,0.677282,1.0,552e4e74dd0a4ebbb3e6915958f8430d
3,hier_clinical_lightgbm_lightgbm_log_mr0p9,lightgbm,lightgbm,log,0.9,0,0.0,1.0,1.0,0.382967,0.365443,0.677282,1.0,87f92b9031cc49e0bf562b6cf7e648d4
4,hier_clinical_xgboost_xgboost_balanced_mr0p9,xgboost,xgboost,balanced,0.9,0,0.0,1.0,1.0,0.366180,0.366741,0.653928,1.0,8d1e30a986f24aa5b209058e9322e23c
5,hier_clinical_lightgbm_xgboost_balanced_mr0p9,lightgbm,xgboost,balanced,0.9,0,0.0,1.0,1.0,0.366180,0.366741,0.653928,1.0,a18fd80b3f0a4c05828b60028630aeae
6,hier_clinical_xgboost_lightgbm_sqrt_mr0p9,xgboost,lightgbm,sqrt,0.9,0,0.0,1.0,1.0,0.358202,0.342859,0.668790,1.0,ff66da40b0be4ec7840659128e368c3c
7,hier_clinical_lightgbm_lightgbm_sqrt_mr0p9,lightgbm,lightgbm,sqrt,0.9,0,0.0,1.0,1.0,0.358202,0.342859,0.668790,1.0,ada1bc8125dd4bd3ad6de5519a6be6da
8,hier_clinical_xgboost_xgboost_sqrt_mr0p9,xgboost,xgboost,sqrt,0.9,0,0.0,1.0,1.0,0.349547,0.331661,0.653928,1.0,3c334390b37f4556a2d2dec42b954d45
9,hier_clinical_lightgbm_xgboost_sqrt_mr0p9,lightgbm,xgboost,sqrt,0.9,0,0.0,1.0,1.0,0.349547,0.331661,0.653928,1.0,1138daca43474dd880463275b50b9282


In [11]:
if len(err_df) > 0:
    print("Errores encontrados:")
    display(err_df)

## Comparación con 5.0 (solo genes)

In [12]:
print("=== Mejor modelo 5.2 (genes + clínicas + clusters como features) ===")
best = res_df.iloc[0]
print(f"  Stage1 clf:        {best['s1_clf']}")
print(f"  Stage2 clf:        {best['s2_clf']}")
print(f"  Stage2 weighting:  {best['s2_weighting']}")
print(f"  test_cancer_fn:    {best['test_cancer_fn']}")
print(f"  test_cancer_recall:{best['test_cancer_recall']:.4f}")
print(f"  test_f1_macro:     {best['test_f1_macro']:.4f}")
print(f"  test_bal_acc:      {best['test_balanced_accuracy']:.4f}")
print(f"  test_accuracy:     {best['test_accuracy']:.4f}")

print("\n--- Comparar con 5.0 ejecutando ambos notebooks ---")
print("Métricas de 5.0 se pueden consultar en MLflow o en el notebook 5.0.")

=== Mejor modelo 5.2 (genes + clínicas + clusters como features) ===
  Stage1 clf:        xgboost
  Stage2 clf:        lightgbm
  Stage2 weighting:  balanced
  test_cancer_fn:    0
  test_cancer_recall:1.0000
  test_f1_macro:     0.4143
  test_bal_acc:      0.3910
  test_accuracy:     0.6879

--- Comparar con 5.0 ejecutando ambos notebooks ---
Métricas de 5.0 se pueden consultar en MLflow o en el notebook 5.0.


## Entrenamiento final del mejor modelo

Entrenamos con guardado completo: bundle local, plots, MLflow.

In [13]:
best = res_df.iloc[0].to_dict()
print("Mejor configuración:")
for k, v in best.items():
    if k not in ["mlflow_run_id", "error"]:
        print(f"  {k}: {v}")

Mejor configuración:
  model_name: hier_clinical_xgboost_lightgbm_balanced_mr0p9
  s1_clf: xgboost
  s2_clf: lightgbm
  s2_weighting: balanced
  s1_min_recall: 0.9
  test_cancer_fn: 0
  test_cancer_fnr: 0.0
  test_cancer_recall: 1.0
  test_cancer_specificity: 1.0
  test_f1_macro: 0.41426868954509566
  test_balanced_accuracy: 0.39103123330962664
  test_accuracy: 0.6878980891719745
  test_cancer_roc_auc: 1.0


In [14]:
BEST_CONFIG = HierarchicalTrainConfig(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    class_group_col="Class_group",
    patient_group_col="Patient_group",
    nonmalignant_label="nonMalignant",
    malignant_label="Malignant",
    model_name="hierarchical_clinical_final",
    model_version="v0.5.0",
    stage1_clf_name=best["s1_clf"],
    stage1_clf_params=STAGE1_CLASSIFIERS_DICT[best["s1_clf"]],
    stage1_min_recall=float(best["s1_min_recall"]),
    stage2_clf_name=best["s2_clf"],
    stage2_clf_params=STAGE2_CLASSIFIERS_DICT[best["s2_clf"]],
    stage2_class_weighting=best["s2_weighting"],
    use_pca=False,
    var_quantile=0.15,
    selector_on_log=False,
    variance_filter_threshold=1e-6,
    cv_splits=8,
    skip_cv_for_sweep=False,
    random_state=42,
    experiment_name="gse183635_hierarchical_clinical",
    save_local_bundle=True,
    save_plots=True,
    mlflow_log_artifacts=True,
    mlflow_log_model=True,
    output_figures_dir="reports/figures/hierarchical_clinical",
    # --- Features clínicas ---
    cluster_as_feature=True,
    malignant_cluster_col="mal_cluster",
    nm_cluster_col="nm_cluster",
    age_col="Age",
    sex_col="Sex",
)

final_result = run_hierarchical_training(BEST_CONFIG, feature_cols=gene_cols)

print("\nMÉTRICAS MODELO FINAL")

print("\n--- Métricas de Test ---")
tm = final_result["test_metrics"]
print(f"  Accuracy:           {tm['accuracy']:.4f}")
print(f"  Balanced Accuracy:  {tm['balanced_accuracy']:.4f}")
print(f"  F1 Macro:           {tm['f1_macro']:.4f}")
print(f"  F1 Weighted:        {tm['f1_weighted']:.4f}")
print(f"  Cancer Recall:      {tm['cancer_recall_sensitivity']:.4f}")
print(f"  Cancer Specificity: {tm['cancer_specificity']:.4f}")
print(f"  Cancer FN:          {tm['cancer_fn']}")
print(f"  Cancer ROC AUC:     {tm['cancer_roc_auc']:.4f}")
print(f"  Cancer PR AUC:      {tm['cancer_pr_auc']:.4f}")
print(f"  Stage1 Threshold:   {tm['stage1_threshold']:.4f}")
print(f"\n  Bundle: {final_result['bundle_dir']}")

/workspaces/TFM/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/workspaces/TFM/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/workspaces/TFM/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/workspaces/TFM/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/workspaces/TFM/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.w


MÉTRICAS MODELO FINAL

--- Métricas de Test ---
  Accuracy:           0.6730
  Balanced Accuracy:  0.3740
  F1 Macro:           0.3993
  F1 Weighted:        0.6471
  Cancer Recall:      0.9571
  Cancer Specificity: 1.0000
  Cancer FN:          14
  Cancer ROC AUC:     1.0000
  Cancer PR AUC:      1.0000
  Stage1 Threshold:   0.9564

  Bundle: /workspaces/TFM/models/hierarchical_clinical_final/v0.5.0
